# Edge Type Correlation Analysis with Degree Coloring

This notebook analyzes validation correlations for individual edge types using two approaches:
1. Held-out permutation validation (permutations 21-30)
2. TRUE empirical frequency validation (notebook 3 output)

## Methodology

**Validation Approach 1: Held-Out Permutations**
- Tests model generalization to held-out permutations 21-30
- Uses same biased sampling as training (edges + 10% non-edges)
- Validates models trained on permutations 1-20

**Validation Approach 2: TRUE Empirical Frequencies**
- Compares against notebook 3 empirical frequencies (ALL permutations)
- Makes results comparable to notebook 5 analytical benchmarks
- Tests accuracy vs ground truth probabilities

## Key Questions

1. How well do models generalize to held-out permutations?
2. How accurate are models vs TRUE empirical probabilities?
3. Why is analytical correlation lower with biased sampling?

In [ ]:
# Papermill parameters
validation_perm_range = (21, 31)
model_types = ["rf", "poly", "analytical"]
min_edge_count = 1000
color_by_log = True
save_plots = True

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
from scipy.stats import pearsonr
import joblib
import warnings
from collections import defaultdict
import json
import gc
warnings.filterwarnings('ignore')

repo_dir = Path.cwd().parent
data_dir = repo_dir / 'data'
models_dir = repo_dir / 'results' / 'null_models'
results_dir = repo_dir / 'results' / 'edge_correlation_analysis'
results_dir.mkdir(parents=True, exist_ok=True)

print(f"Repository directory: {repo_dir}")
print(f"Data directory: {data_dir}")
print(f"Models directory: {models_dir}")
print(f"Results will be saved to: {results_dir}")
print(f"Validation permutations: {validation_perm_range[0]}-{validation_perm_range[1]-1}")

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

## 1. Discover Available Edge Types

In [ ]:
def discover_edge_types():
    """
    Discover available edge types by checking both data files and trained models.
    
    Returns
    -------
    list of str
        Edge types with available trained models
    """
    edge_types = set()
    
    edges_dir = data_dir / 'edges'
    if edges_dir.exists():
        for edge_file in edges_dir.glob('*.sparse.npz'):
            edge_type = edge_file.stem
            edge_types.add(edge_type)
    
    perm_edges_dir = data_dir / 'permutations' / '000.hetmat' / 'edges'
    if perm_edges_dir.exists():
        for edge_file in perm_edges_dir.glob('*.sparse.npz'):
            edge_type = edge_file.stem
            edge_types.add(edge_type)
    
    available_edge_types = []
    for edge_type in sorted(edge_types):
        has_models = False
        for model_type in model_types:
            if model_type == 'analytical':
                continue
            model_file = models_dir / f'{edge_type}_{model_type}_null.pkl'
            if model_file.exists():
                has_models = True
                break
        
        if has_models:
            available_edge_types.append(edge_type)
    
    return available_edge_types

available_edge_types = discover_edge_types()
print(f"\nDiscovered {len(available_edge_types)} edge types with trained models:")
for i, edge_type in enumerate(available_edge_types):
    print(f"  {i+1:2d}. {edge_type}")

## 2. Load Models and Validate Edge Types

In [ ]:
def load_edge_model(edge_type, model_type):
    """
    Load trained model for specific edge type and model type.
    
    Parameters
    ----------
    edge_type : str
        Edge type (e.g., 'CbG')
    model_type : str
        Model type ('rf' or 'poly')
    
    Returns
    -------
    object or dict or None
        Loaded model or None if not found
    """
    try:
        if model_type == 'rf':
            model_file = models_dir / f'{edge_type}_rf_null.pkl'
            if model_file.exists():
                return joblib.load(model_file)

        elif model_type == 'poly':
            model_file = models_dir / f'{edge_type}_poly_null.pkl'
            features_file = models_dir / f'{edge_type}_poly_features.pkl'
            if model_file.exists() and features_file.exists():
                model = joblib.load(model_file)
                features = joblib.load(features_file)
                return {'model': model, 'features': features}

    except Exception as e:
        print(f"    Error loading {model_type} model for {edge_type}: {e}")

    return None

def validate_edge_type(edge_type):
    """
    Check if edge type has sufficient data and models for analysis.
    
    Parameters
    ----------
    edge_type : str
        Edge type to validate
    
    Returns
    -------
    tuple of (bool, str)
        (is_valid, message)
    """
    edge_file = data_dir / 'edges' / f'{edge_type}.sparse.npz'
    if not edge_file.exists():
        edge_file = data_dir / 'permutations' / '000.hetmat' / 'edges' / f'{edge_type}.sparse.npz'

    if not edge_file.exists():
        return False, "No data file found"

    try:
        matrix = sp.load_npz(str(edge_file))
        edge_count = matrix.nnz
        del matrix
        gc.collect()

        if edge_count < min_edge_count:
            return False, f"Insufficient edges ({edge_count:,} < {min_edge_count:,})"

    except Exception as e:
        return False, f"Error loading data: {e}"

    models_available = []
    for model_type in ['rf', 'poly']:
        model = load_edge_model(edge_type, model_type)
        if model is not None:
            models_available.append(model_type)

    if not models_available:
        return False, "No trained models found"

    return True, f"Valid ({edge_count:,} edges, models: {models_available})"

print("\nValidating edge types...")
valid_edge_types = []
validation_results = {}

for edge_type in available_edge_types:
    is_valid, message = validate_edge_type(edge_type)
    validation_results[edge_type] = message

    if is_valid:
        valid_edge_types.append(edge_type)
        print(f"  VALID {edge_type}: {message}")
    else:
        print(f"  SKIPPED {edge_type}: {message}")

print(f"\n{len(valid_edge_types)} edge types ready for analysis")

## 3. Extract Empirical Frequencies from Validation Permutations

In [ ]:
def load_null_edge_frequencies(edge_type, perm_ids, negative_sample_ratio=0.1):
    """
    Load and aggregate edge frequencies from null permutations.
    
    Uses same methodology as notebook 13 for consistency.
    Samples edges + negative samples, then aggregates by degree pair.
    
    Parameters
    ----------
    edge_type : str
        Edge type (e.g., 'CbG')
    perm_ids : list or range
        Permutation IDs to load
    negative_sample_ratio : float, default=0.1
        Ratio of negative samples to positive samples
        
    Returns
    -------
    pd.DataFrame
        DataFrame with columns: source_degree, target_degree, edge_probability
    """
    all_pairs = []
    
    for perm_id in perm_ids:
        edge_file = data_dir / 'permutations' / f'{perm_id:03d}.hetmat' / 'edges' / f'{edge_type}.sparse.npz'
        
        if not edge_file.exists():
            continue
        
        matrix = sp.load_npz(str(edge_file))
        source_degrees = np.array(matrix.sum(axis=1)).flatten()
        target_degrees = np.array(matrix.sum(axis=0)).flatten()
        
        for i, j in zip(*matrix.nonzero()):
            all_pairs.append({
                'source_degree': int(source_degrees[i]),
                'target_degree': int(target_degrees[j]),
                'edge_exists': 1
            })
        
        n_nodes_source, n_nodes_target = matrix.shape
        n_negatives = int(matrix.nnz * negative_sample_ratio)
        
        sampled = 0
        attempts = 0
        max_attempts = n_negatives * 10
        
        while sampled < n_negatives and attempts < max_attempts:
            i = np.random.randint(0, n_nodes_source)
            j = np.random.randint(0, n_nodes_target)
            
            if matrix[i, j] == 0 and source_degrees[i] > 0 and target_degrees[j] > 0:
                all_pairs.append({
                    'source_degree': int(source_degrees[i]),
                    'target_degree': int(target_degrees[j]),
                    'edge_exists': 0
                })
                sampled += 1
            attempts += 1
        
        del matrix
        gc.collect()
    
    df = pd.DataFrame(all_pairs)
    
    if len(df) == 0:
        return pd.DataFrame(columns=['source_degree', 'target_degree', 'edge_probability'])
    
    null_freq = df.groupby(['source_degree', 'target_degree']).agg({
        'edge_exists': ['sum', 'count', 'mean']
    }).reset_index()
    
    null_freq.columns = ['source_degree', 'target_degree', 'edge_count', 'total_count', 'edge_probability']
    
    return null_freq

def collect_empirical_data(edge_type):
    """
    Collect empirical edge frequencies across validation permutations.
    
    Parameters
    ----------
    edge_type : str
        Edge type to process
    
    Returns
    -------
    pd.DataFrame or None
        DataFrame with empirical frequencies or None if no data
    """
    print(f"\n  Processing {edge_type}...")
    
    val_freq = load_null_edge_frequencies(
        edge_type,
        range(validation_perm_range[0], validation_perm_range[1]),
        negative_sample_ratio=0.1
    )
    
    if len(val_freq) == 0:
        print(f"    No data found for {edge_type}")
        return None
    
    val_freq['degree_product'] = val_freq['source_degree'] * val_freq['target_degree']
    val_freq['log_degree_product'] = np.log10(val_freq['degree_product'] + 1)
    
    print(f"    Collected {len(val_freq):,} unique degree pairs")
    print(f"    Edge probability range: [{val_freq['edge_probability'].min():.6f}, {val_freq['edge_probability'].max():.6f}]")
    print(f"    Edge probability variance: {val_freq['edge_probability'].var():.6f}")
    
    val_freq.rename(columns={'edge_probability': 'empirical_frequency'}, inplace=True)
    
    return val_freq

print("Collecting empirical frequencies from validation permutations...")
empirical_data = {}

for edge_type in valid_edge_types:
    data = collect_empirical_data(edge_type)
    if data is not None:
        empirical_data[edge_type] = data

print(f"\nCollected empirical data for {len(empirical_data)} edge types")

## 4. Compute Analytical Prior Predictions

In [ ]:
def compute_analytical_prior(edge_type, empirical_df, data_dir):
    """
    Compute XSwap analytical prior predictions.
    
    Formula from Himmelstein & Baranzini (2015):
        P(u,v|G) = (u × v) / sqrt[(u×v)^2 + (m - u - v + 1)^2]
    
    where:
        u = source node degree
        v = target node degree
        m = total number of edges in the graph
    
    Parameters
    ----------
    edge_type : str
        Edge type (e.g., 'CbG')
    empirical_df : pd.DataFrame
        DataFrame with 'source_degree' and 'target_degree' columns
    data_dir : Path
        Path to data directory
    
    Returns
    -------
    np.ndarray
        Predicted edge probabilities using analytical formula
    
    References
    ----------
    Himmelstein DS, Baranzini SE (2015) Heterogeneous Network Edge Prediction:
    A Data Integration Approach to Prioritize Disease-Associated Genes.
    PLoS Comput Biol 11(7): e1004259. doi:10.1371/journal.pcbi.1004259
    """
    edge_file = data_dir / 'edges' / f'{edge_type}.sparse.npz'
    if not edge_file.exists():
        edge_file = data_dir / 'permutations' / '000.hetmat' / 'edges' / f'{edge_type}.sparse.npz'
    
    matrix = sp.load_npz(str(edge_file))
    m = matrix.nnz
    del matrix
    gc.collect()
    
    predictions = []
    for _, row in empirical_df.iterrows():
        u = row['source_degree']
        v = row['target_degree']
        
        uv = u * v
        denominator_term = (m - u - v + 1)
        denominator = np.sqrt(uv**2 + denominator_term**2)
        
        if denominator > 0:
            p_analytical = uv / denominator
        else:
            p_analytical = 0.0
        
        p_analytical = np.clip(p_analytical, 0.0, 1.0)
        predictions.append(p_analytical)
    
    return np.array(predictions)

print("XSwap analytical prior function defined")

## 5. Compute Model Predictions and Correlations

In [ ]:
def compute_model_predictions(edge_type, empirical_df, model_type):
    """
    Compute model predictions for empirical degree pairs.
    
    Parameters
    ----------
    edge_type : str
        Edge type
    empirical_df : pd.DataFrame
        DataFrame with degree pairs
    model_type : str
        Model type ('rf', 'poly', or 'analytical')
    
    Returns
    -------
    np.ndarray or None
        Predictions or None if error
    """
    try:
        if model_type == 'analytical':
            predictions = compute_analytical_prior(edge_type, empirical_df, data_dir)
            if predictions is not None and len(predictions) > 0:
                print(f"    Analytical: {len(predictions)} predictions, range [{predictions.min():.6f}, {predictions.max():.6f}], std={predictions.std():.6f}")
            return predictions

        model = load_edge_model(edge_type, model_type)
        if model is None:
            return None

        degree_pairs = empirical_df[['source_degree', 'target_degree']].values

        if model_type == 'rf':
            predictions = model.predict(degree_pairs)
        elif model_type == 'poly':
            X_poly = model['features'].transform(degree_pairs)
            predictions = model['model'].predict(X_poly)
        else:
            return None

        predictions = np.clip(predictions, 0, 1)

        return predictions

    except Exception as e:
        print(f"    Error computing {model_type} predictions for {edge_type}: {e}")
        import traceback
        traceback.print_exc()
        return None

def compute_correlation(edge_type, empirical_df):
    """
    Compute correlations between empirical and predicted frequencies.
    
    Parameters
    ----------
    edge_type : str
        Edge type
    empirical_df : pd.DataFrame
        DataFrame with empirical frequencies
    
    Returns
    -------
    dict
        Correlation results for all model types
    """
    results = {
        'edge_type': edge_type,
        'n_pairs': len(empirical_df),
        'mean_source_degree': empirical_df['source_degree'].mean(),
        'mean_target_degree': empirical_df['target_degree'].mean(),
        'mean_degree_product': empirical_df['degree_product'].mean(),
        'log_mean_degree_product': np.log10(empirical_df['degree_product'].mean() + 1)
    }

    for model_type in model_types:
        predictions = compute_model_predictions(edge_type, empirical_df, model_type)

        if predictions is not None and len(predictions) > 0:
            empirical_values = empirical_df['empirical_frequency'].values
            
            if len(empirical_values) > 1 and empirical_values.std() > 0 and predictions.std() > 0:
                try:
                    corr, p_val = pearsonr(empirical_values, predictions)
                    results[f'{model_type}_correlation'] = corr
                    results[f'{model_type}_p_value'] = p_val
                    results[f'{model_type}_n_pairs'] = len(empirical_values)
                except Exception as e:
                    print(f"    Warning: Could not compute {model_type} correlation: {e}")
                    results[f'{model_type}_correlation'] = np.nan
                    results[f'{model_type}_p_value'] = np.nan
                    results[f'{model_type}_n_pairs'] = len(empirical_values)
            else:
                print(f"    Warning: {model_type} has insufficient variance for correlation")
                print(f"      Empirical std: {empirical_values.std():.6f}, Predicted std: {predictions.std():.6f}")
                results[f'{model_type}_correlation'] = np.nan
                results[f'{model_type}_p_value'] = np.nan
                results[f'{model_type}_n_pairs'] = len(empirical_values)
        else:
            results[f'{model_type}_correlation'] = np.nan
            results[f'{model_type}_p_value'] = np.nan
            results[f'{model_type}_n_pairs'] = 0

    return results

print("\nComputing model predictions and correlations...")
correlation_results = []

for edge_type in empirical_data.keys():
    print(f"\n  Processing {edge_type}...")
    empirical_df = empirical_data[edge_type]

    result = compute_correlation(edge_type, empirical_df)
    correlation_results.append(result)

    print(f"    Pairs: {result['n_pairs']:,}")
    for model_type in model_types:
        corr = result.get(f'{model_type}_correlation', np.nan)
        if not np.isnan(corr):
            print(f"    {model_type.upper()} correlation: {corr:.4f}")
        else:
            print(f"    {model_type.upper()} correlation: N/A")

results_df = pd.DataFrame(correlation_results)
print(f"\nComputed correlations for {len(results_df)} edge types")

## 6. Compare Against TRUE Empirical Frequencies (Notebook 3)

This section compares model predictions against TRUE empirical frequencies computed in notebook 3 across ALL permutations, making results directly comparable to notebook 5 analytical benchmarks.

**Key Difference:**
- Section 5: Compares against held-out permutations 21-30 with biased sampling (tests generalization)
- Section 6: Compares against TRUE probabilities from notebook 3 (tests accuracy vs ground truth)

In [ ]:
def load_true_empirical_frequencies(edge_type):
    """
    Load TRUE empirical frequencies from notebook 3 output.
    
    These are computed across ALL permutations using:
        frequency = edges_observed / total_possible_node_pairs
    
    Parameters
    ----------
    edge_type : str
        Edge type (e.g., 'CbG')
    
    Returns
    -------
    pd.DataFrame or None
        DataFrame with TRUE empirical frequencies or None if not found
    """
    empirical_freq_file = repo_dir / 'results' / 'empirical_edge_frequencies' / f'edge_frequency_by_degree_{edge_type}.csv'
    
    if not empirical_freq_file.exists():
        print(f"    Warning: TRUE empirical frequency file not found for {edge_type}")
        print(f"    Expected: {empirical_freq_file}")
        return None
    
    try:
        true_empirical_df = pd.read_csv(empirical_freq_file)
        
        if 'empirical_frequency' not in true_empirical_df.columns:
            true_empirical_df.rename(columns={'frequency': 'empirical_frequency'}, inplace=True)
        
        true_empirical_df['degree_product'] = (true_empirical_df['source_degree'] * 
                                               true_empirical_df['target_degree'])
        true_empirical_df['log_degree_product'] = np.log10(true_empirical_df['degree_product'] + 1)
        
        return true_empirical_df
        
    except Exception as e:
        print(f"    Error loading TRUE empirical frequencies for {edge_type}: {e}")
        return None

def compute_correlation_vs_true_empirical(edge_type):
    """
    Compute correlations between model predictions and TRUE empirical frequencies.
    
    Makes results directly comparable to notebook 5.
    
    Parameters
    ----------
    edge_type : str
        Edge type
    
    Returns
    -------
    dict or None
        Correlation results or None if no data
    """
    true_empirical_df = load_true_empirical_frequencies(edge_type)
    
    if true_empirical_df is None or len(true_empirical_df) == 0:
        return None
    
    print(f"    Loaded {len(true_empirical_df):,} degree pairs from notebook 3")
    print(f"    TRUE empirical frequency range: [{true_empirical_df['empirical_frequency'].min():.6f}, {true_empirical_df['empirical_frequency'].max():.6f}]")
    print(f"    Mean TRUE empirical: {true_empirical_df['empirical_frequency'].mean():.6f}")
    
    results = {
        'edge_type': edge_type,
        'n_pairs_true_empirical': len(true_empirical_df),
        'mean_source_degree': true_empirical_df['source_degree'].mean(),
        'mean_target_degree': true_empirical_df['target_degree'].mean(),
        'mean_degree_product': true_empirical_df['degree_product'].mean(),
        'log_mean_degree_product': np.log10(true_empirical_df['degree_product'].mean() + 1),
        'mean_true_empirical_freq': true_empirical_df['empirical_frequency'].mean()
    }
    
    for model_type in model_types:
        predictions = compute_model_predictions(edge_type, true_empirical_df, model_type)
        
        if predictions is not None and len(predictions) > 0:
            empirical_values = true_empirical_df['empirical_frequency'].values
            
            if len(empirical_values) > 1 and empirical_values.std() > 0 and predictions.std() > 0:
                try:
                    corr, p_val = pearsonr(empirical_values, predictions)
                    results[f'{model_type}_correlation_true'] = corr
                    results[f'{model_type}_p_value_true'] = p_val
                except Exception as e:
                    print(f"    Warning: Could not compute {model_type} correlation: {e}")
                    results[f'{model_type}_correlation_true'] = np.nan
                    results[f'{model_type}_p_value_true'] = np.nan
            else:
                print(f"    Warning: {model_type} has insufficient variance")
                results[f'{model_type}_correlation_true'] = np.nan
                results[f'{model_type}_p_value_true'] = np.nan
        else:
            results[f'{model_type}_correlation_true'] = np.nan
            results[f'{model_type}_p_value_true'] = np.nan
    
    return results

print("\nComputing correlations vs TRUE empirical frequencies (notebook 3)...")
true_empirical_results = []

for edge_type in valid_edge_types:
    print(f"\n  Processing {edge_type}...")
    
    result = compute_correlation_vs_true_empirical(edge_type)
    
    if result is not None:
        true_empirical_results.append(result)
        
        print(f"    Pairs: {result['n_pairs_true_empirical']:,}")
        for model_type in model_types:
            corr = result.get(f'{model_type}_correlation_true', np.nan)
            if not np.isnan(corr):
                print(f"    {model_type.upper()} correlation vs TRUE empirical: {corr:.4f}")
            else:
                print(f"    {model_type.upper()} correlation vs TRUE empirical: N/A")

true_empirical_df = pd.DataFrame(true_empirical_results)
print(f"\nComputed TRUE empirical correlations for {len(true_empirical_df)} edge types")

## 7. Summary Statistics

In [ ]:
if len(results_df) > 0:
    print("\n" + "="*80)
    print("EDGE TYPE CORRELATION ANALYSIS SUMMARY")
    print("="*80)
    
    print("\n" + "="*80)
    print("VALIDATION APPROACH 1: Held-Out Permutations (21-30) - Biased Sampling")
    print("="*80)
    print("Tests: Model generalization to held-out permutations")
    
    total_pairs = results_df['n_pairs'].sum()
    print(f"\nProcessed {len(results_df)} edge types with {total_pairs:,} total degree pairs")
    
    rf_valid = results_df.dropna(subset=['rf_correlation'])
    poly_valid = results_df.dropna(subset=['poly_correlation'])
    analytical_valid = results_df.dropna(subset=['analytical_correlation'])
    
    if len(rf_valid) > 0:
        print(f"\nRandom Forest Results ({len(rf_valid)} edge types):")
        print(f"  Mean correlation: {rf_valid['rf_correlation'].mean():.4f}")
        print(f"  Std correlation: {rf_valid['rf_correlation'].std():.4f}")
        print(f"  Best edge type: {rf_valid.loc[rf_valid['rf_correlation'].idxmax(), 'edge_type']} (r = {rf_valid['rf_correlation'].max():.4f})")
        print(f"  Worst edge type: {rf_valid.loc[rf_valid['rf_correlation'].idxmin(), 'edge_type']} (r = {rf_valid['rf_correlation'].min():.4f})")
    
    if len(poly_valid) > 0:
        print(f"\nPolynomial LogReg Results ({len(poly_valid)} edge types):")
        print(f"  Mean correlation: {poly_valid['poly_correlation'].mean():.4f}")
        print(f"  Std correlation: {poly_valid['poly_correlation'].std():.4f}")
        print(f"  Best edge type: {poly_valid.loc[poly_valid['poly_correlation'].idxmax(), 'edge_type']} (r = {poly_valid['poly_correlation'].max():.4f})")
        print(f"  Worst edge type: {poly_valid.loc[poly_valid['poly_correlation'].idxmin(), 'edge_type']} (r = {poly_valid['poly_correlation'].min():.4f})")
    
    if len(analytical_valid) > 0:
        print(f"\nXSwap Analytical Prior Results ({len(analytical_valid)} edge types):")
        print(f"  Mean correlation: {analytical_valid['analytical_correlation'].mean():.4f}")
        print(f"  Std correlation: {analytical_valid['analytical_correlation'].std():.4f}")
        print(f"  Best edge type: {analytical_valid.loc[analytical_valid['analytical_correlation'].idxmax(), 'edge_type']} (r = {analytical_valid['analytical_correlation'].max():.4f})")
        print(f"  Worst edge type: {analytical_valid.loc[analytical_valid['analytical_correlation'].idxmin(), 'edge_type']} (r = {analytical_valid['analytical_correlation'].min():.4f})")

if len(true_empirical_df) > 0:
    print("\n" + "="*80)
    print("VALIDATION APPROACH 2: TRUE Empirical Frequencies (Notebook 3)")
    print("="*80)
    print("Tests: Model accuracy vs ground truth (comparable to Notebook 5)")
    
    total_pairs_true = true_empirical_df['n_pairs_true_empirical'].sum()
    print(f"\nProcessed {len(true_empirical_df)} edge types with {total_pairs_true:,} total degree pairs")
    
    rf_valid_true = true_empirical_df.dropna(subset=['rf_correlation_true'])
    poly_valid_true = true_empirical_df.dropna(subset=['poly_correlation_true'])
    analytical_valid_true = true_empirical_df.dropna(subset=['analytical_correlation_true'])
    
    if len(rf_valid_true) > 0:
        print(f"\nRandom Forest Results ({len(rf_valid_true)} edge types):")
        print(f"  Mean correlation: {rf_valid_true['rf_correlation_true'].mean():.4f}")
        print(f"  Std correlation: {rf_valid_true['rf_correlation_true'].std():.4f}")
        print(f"  Best edge type: {rf_valid_true.loc[rf_valid_true['rf_correlation_true'].idxmax(), 'edge_type']} (r = {rf_valid_true['rf_correlation_true'].max():.4f})")
        print(f"  Worst edge type: {rf_valid_true.loc[rf_valid_true['rf_correlation_true'].idxmin(), 'edge_type']} (r = {rf_valid_true['rf_correlation_true'].min():.4f})")
    
    if len(poly_valid_true) > 0:
        print(f"\nPolynomial LogReg Results ({len(poly_valid_true)} edge types):")
        print(f"  Mean correlation: {poly_valid_true['poly_correlation_true'].mean():.4f}")
        print(f"  Std correlation: {poly_valid_true['poly_correlation_true'].std():.4f}")
        print(f"  Best edge type: {poly_valid_true.loc[poly_valid_true['poly_correlation_true'].idxmax(), 'edge_type']} (r = {poly_valid_true['poly_correlation_true'].max():.4f})")
        print(f"  Worst edge type: {poly_valid_true.loc[poly_valid_true['poly_correlation_true'].idxmin(), 'edge_type']} (r = {poly_valid_true['poly_correlation_true'].min():.4f})")
    
    if len(analytical_valid_true) > 0:
        print(f"\nXSwap Analytical Prior Results ({len(analytical_valid_true)} edge types):")
        print(f"  Mean correlation: {analytical_valid_true['analytical_correlation_true'].mean():.4f}")
        print(f"  Std correlation: {analytical_valid_true['analytical_correlation_true'].std():.4f}")
        print(f"  Best edge type: {analytical_valid_true.loc[analytical_valid_true['analytical_correlation_true'].idxmax(), 'edge_type']} (r = {analytical_valid_true['analytical_correlation_true'].max():.4f})")
        print(f"  Worst edge type: {analytical_valid_true.loc[analytical_valid_true['analytical_correlation_true'].idxmin(), 'edge_type']} (r = {analytical_valid_true['analytical_correlation_true'].min():.4f})")
        
    print("\n" + "="*80)
    print("KEY INSIGHT: Analytical Correlation Comparison")
    print("="*80)
    print(f"Analytical vs Held-Out Perms:   {analytical_valid['analytical_correlation'].mean():.4f}")
    print(f"Analytical vs TRUE Empirical:    {analytical_valid_true['analytical_correlation_true'].mean():.4f}")
    print(f"\nDifference: {analytical_valid_true['analytical_correlation_true'].mean() - analytical_valid['analytical_correlation'].mean():.4f}")
    print("\nThis difference shows the impact of comparing against biased sampling")
    print("vs TRUE probabilities. The TRUE empirical comparison should match Notebook 5.")

print(f"\n" + "="*80)

In [ ]:
## 9. Save Results

In [ ]:
# Visualization 3: Correlation Heatmap by Edge Type
if len(true_empirical_df) > 0 and save_plots:
    # Sort edge types by RF correlation (worst to best)
    edge_types_sorted = true_empirical_df.sort_values('rf_correlation_true')['edge_type'].values
    
    # Create matrix for heatmap
    correlation_matrix = []
    for edge_type in edge_types_sorted:
        row_data = true_empirical_df[true_empirical_df['edge_type'] == edge_type]
        if len(row_data) > 0:
            correlation_matrix.append([
                row_data['rf_correlation_true'].values[0],
                row_data['poly_correlation_true'].values[0],
                row_data['analytical_correlation_true'].values[0]
            ])
    
    correlation_matrix = np.array(correlation_matrix)
    
    # Create heatmap
    fig, ax = plt.subplots(1, 1, figsize=(9, 14))
    
    im = ax.imshow(correlation_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    
    # Set ticks
    ax.set_xticks(np.arange(3))
    ax.set_yticks(np.arange(len(edge_types_sorted)))
    ax.set_xticklabels(['RF', 'Poly', 'Analytical'], fontsize=12, fontweight='bold')
    ax.set_yticklabels(edge_types_sorted, fontsize=10)
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Correlation vs TRUE Empirical', rotation=270, labelpad=25, 
                   fontsize=12, fontweight='bold')
    
    # Add text annotations
    for i in range(len(edge_types_sorted)):
        for j in range(3):
            color = 'white' if correlation_matrix[i, j] < 0.5 else 'black'
            text = ax.text(j, i, f'{correlation_matrix[i, j]:.2f}',
                          ha="center", va="center", color=color, fontsize=9, fontweight='bold')
    
    # Add horizontal line to separate poor RF performers
    poor_threshold_idx = np.sum(correlation_matrix[:, 0] < 0.5)
    if poor_threshold_idx > 0:
        ax.axhline(y=poor_threshold_idx - 0.5, color='red', linestyle='--', linewidth=2)
        ax.text(2.5, poor_threshold_idx - 0.5, 'RF r < 0.5\n(POOR)', 
               va='center', fontsize=10, fontweight='bold', color='red',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    
    ax.set_title('CORRELATION vs TRUE EMPIRICAL BY EDGE TYPE\n(Sorted by RF Performance - Worst to Best)', 
                 fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.savefig(results_dir / 'correlation_heatmap_by_edge_type.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Saved correlation heatmap for {len(edge_types_sorted)} edge types")

In [ ]:
# Visualization 2: Scatter Plots for Sample Edge Types
if len(true_empirical_df) > 0 and save_plots:
    # Select 4 representative edge types (small, medium, large, largest)
    sample_edge_types = ['CtD', 'CbG', 'AeG', 'GpBP']
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    plot_idx = 0
    for edge_type in sample_edge_types:
        # Check if edge type exists in results
        if edge_type not in true_empirical_df['edge_type'].values:
            continue
        
        # Get TRUE empirical data for this edge type
        true_emp_data = load_true_empirical_frequencies(edge_type)
        
        if true_emp_data is None or len(true_emp_data) == 0:
            continue
        
        y_true = true_emp_data['empirical_frequency'].values
        degree_pairs = true_emp_data[['source_degree', 'target_degree']].values
        
        # Get RF predictions
        rf_model = load_edge_model(edge_type, 'rf')
        if rf_model is not None:
            rf_pred = rf_model.predict(degree_pairs)
            rf_pred = np.clip(rf_pred, 0, 1)
            rf_corr, _ = pearsonr(y_true, rf_pred)
            
            # RF scatter
            ax = axes[plot_idx]
            ax.scatter(y_true, rf_pred, alpha=0.3, s=10, c='#EF5350', edgecolors='none')
            ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Perfect Prediction')
            ax.set_xlabel('TRUE Empirical Frequency', fontsize=10, fontweight='bold')
            ax.set_ylabel('RF Predicted', fontsize=10, fontweight='bold')
            ax.set_title(f'{edge_type} - Random Forest\nr = {rf_corr:.3f} (POOR!)', 
                        fontsize=12, fontweight='bold', color='darkred' if rf_corr < 0.6 else 'black')
            ax.grid(True, alpha=0.3)
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            plot_idx += 1
        
        # Get Poly predictions
        poly_model = load_edge_model(edge_type, 'poly')
        if poly_model is not None:
            X_poly = poly_model['features'].transform(degree_pairs)
            poly_pred = poly_model['model'].predict(X_poly)
            poly_pred = np.clip(poly_pred, 0, 1)
            poly_corr, _ = pearsonr(y_true, poly_pred)
            
            # Poly scatter
            ax = axes[plot_idx]
            ax.scatter(y_true, poly_pred, alpha=0.3, s=10, c='#42A5F5', edgecolors='none')
            ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Perfect Prediction')
            ax.set_xlabel('TRUE Empirical Frequency', fontsize=10, fontweight='bold')
            ax.set_ylabel('Poly Predicted', fontsize=10, fontweight='bold')
            ax.set_title(f'{edge_type} - Polynomial\nr = {poly_corr:.3f}', 
                        fontsize=12, fontweight='bold', color='darkred' if poly_corr < 0.7 else 'black')
            ax.grid(True, alpha=0.3)
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            plot_idx += 1
    
    # Hide unused subplots
    for idx in range(plot_idx, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle('MODEL PREDICTIONS vs TRUE EMPIRICAL FREQUENCIES\nShows Poor Correlation for Current Training Approach', 
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(results_dir / 'scatter_plots_sample_edge_types.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Saved scatter plots for {len(sample_edge_types)} sample edge types")

In [ ]:
# Visualization 1: Comparison Bar Chart
if len(results_df) > 0 and len(true_empirical_df) > 0 and save_plots:
    fig, ax = plt.subplots(1, 1, figsize=(12, 7))

    models = ['Random Forest', 'Polynomial LogReg', 'Analytical Prior']
    biased_corrs = [
        results_df['rf_correlation'].mean(),
        results_df['poly_correlation'].mean(),
        results_df['analytical_correlation'].mean()
    ]
    true_corrs = [
        true_empirical_df['rf_correlation_true'].mean(),
        true_empirical_df['poly_correlation_true'].mean(),
        true_empirical_df['analytical_correlation_true'].mean()
    ]

    x = np.arange(len(models))
    width = 0.35

    bars1 = ax.bar(x - width/2, biased_corrs, width, label='Biased Validation (Perms 21-30)', 
                   alpha=0.8, color='#FFA726', edgecolor='black', linewidth=1.5)
    bars2 = ax.bar(x + width/2, true_corrs, width, label='TRUE Empirical (Notebook 3)', 
                   alpha=0.8, color='#42A5F5', edgecolor='black', linewidth=1.5)

    # Annotate bars with values
    for i, (b1, b2) in enumerate(zip(bars1, bars2)):
        ax.text(b1.get_x() + b1.get_width()/2, b1.get_height() + 0.02, 
                f'{biased_corrs[i]:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
        ax.text(b2.get_x() + b2.get_width()/2, b2.get_height() + 0.02,
                f'{true_corrs[i]:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

    ax.set_ylabel('Mean Correlation', fontsize=13, fontweight='bold')
    ax.set_title('VALIDATION PERFORMANCE COMPARISON\nThe Problem We Need to Fix with Notebook 13 Redesign', 
                 fontsize=15, fontweight='bold', pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=12)
    ax.legend(loc='upper right', fontsize=11)
    ax.set_ylim(0, 1.15)
    ax.axhline(y=0.85, color='green', linestyle='--', alpha=0.6, linewidth=2, label='Target (r=0.85)')
    ax.grid(axis='y', alpha=0.3)

    # Annotate the problem - RF catastrophic drop
    ax.annotate('RF CATASTROPHIC\nDROP\n0.87 → 0.43', 
                xy=(0 + width/2, 0.43), xytext=(0 + width/2, 0.65),
                arrowprops=dict(arrowstyle='->', color='red', lw=3),
                fontsize=12, color='red', fontweight='bold',
                ha='center', bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7))

    # Annotate analytical improvement
    ax.annotate('Analytical\nIMPROVES', 
                xy=(2 + width/2, 0.98), xytext=(2 + width/2, 0.75),
                arrowprops=dict(arrowstyle='->', color='darkgreen', lw=2),
                fontsize=10, color='darkgreen', fontweight='bold',
                ha='center')

    plt.tight_layout()
    plt.savefig(results_dir / 'validation_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Saved validation comparison bar chart")

## 8. Visualizations

Visual documentation of the performance problem with current training approach.
Shows why notebook 13 needs to be redesigned to train on binary samples instead of aggregated probabilities.

## 8. Save Results

In [ ]:
if len(results_df) > 0:
    results_file = results_dir / 'edge_correlation_results_permutation_validation.csv'
    results_df.to_csv(results_file, index=False)
    print(f"\nHeld-out permutation validation results saved to: {results_file}")
    
    summary_stats = {
        'analysis_type': 'edge_correlation_analysis_permutation_validation',
        'validation_permutations': list(range(validation_perm_range[0], validation_perm_range[1])),
        'n_edge_types': len(results_df),
        'model_types': model_types,
        'total_degree_pairs': int(results_df['n_pairs'].sum()),
        'rf_mean_correlation': float(results_df['rf_correlation'].mean()) if 'rf_correlation' in results_df.columns else None,
        'poly_mean_correlation': float(results_df['poly_correlation'].mean()) if 'poly_correlation' in results_df.columns else None,
        'analytical_mean_correlation': float(results_df['analytical_correlation'].mean()) if 'analytical_correlation' in results_df.columns else None,
        'edge_types_analyzed': results_df['edge_type'].tolist()
    }
    
    summary_file = results_dir / 'permutation_validation_summary.json'
    with open(summary_file, 'w') as f:
        json.dump(summary_stats, f, indent=2)
    
    print(f"Permutation validation summary saved to: {summary_file}")

if len(true_empirical_df) > 0:
    true_results_file = results_dir / 'edge_correlation_results_true_empirical.csv'
    true_empirical_df.to_csv(true_results_file, index=False)
    print(f"\nTRUE empirical validation results saved to: {true_results_file}")
    
    true_summary_stats = {
        'analysis_type': 'edge_correlation_analysis_true_empirical',
        'empirical_source': 'notebook_3_all_permutations',
        'n_edge_types': len(true_empirical_df),
        'model_types': model_types,
        'total_degree_pairs': int(true_empirical_df['n_pairs_true_empirical'].sum()),
        'rf_mean_correlation': float(true_empirical_df['rf_correlation_true'].mean()) if 'rf_correlation_true' in true_empirical_df.columns else None,
        'poly_mean_correlation': float(true_empirical_df['poly_correlation_true'].mean()) if 'poly_correlation_true' in true_empirical_df.columns else None,
        'analytical_mean_correlation': float(true_empirical_df['analytical_correlation_true'].mean()) if 'analytical_correlation_true' in true_empirical_df.columns else None,
        'mean_true_empirical_freq': float(true_empirical_df['mean_true_empirical_freq'].mean()) if 'mean_true_empirical_freq' in true_empirical_df.columns else None,
        'edge_types_analyzed': true_empirical_df['edge_type'].tolist(),
        'comparable_to': 'notebook_5_analytical_benchmarks'
    }
    
    true_summary_file = results_dir / 'true_empirical_validation_summary.json'
    with open(true_summary_file, 'w') as f:
        json.dump(true_summary_stats, f, indent=2)
    
    print(f"TRUE empirical validation summary saved to: {true_summary_file}")

print(f"\nAnalysis complete! Results saved to: {results_dir}")